# Graph-Enhanced ETA Model

In [6]:
import pandas as pd
import numpy as np

leg_df = pd.read_csv(
    r"D:\Projects\Delivery_ETA\outputs\leg_level_data.csv"
)

node_metrics = pd.read_csv(
    r"D:\Projects\Delivery_ETA\outputs\node_metrics.csv"
)

corridor_metrics = pd.read_csv(
    r"D:\Projects\Delivery_ETA\outputs\corridor_metrics.csv"
)

# Source features
source_features = node_metrics.add_prefix("source_")

model_df = leg_df.merge(
    source_features,
    left_on="source_center",
    right_on="source_hub",
    how="left"
)

# Destination features
destination_features = node_metrics.add_prefix("destination_")

model_df = model_df.merge(
    destination_features,
    left_on="destination_center",
    right_on="destination_hub",
    how="left"
)

# Corridor features
corridor_features = corridor_metrics[
    [
        "source_center",
        "destination_center",
        "trip_count",
        "delay_ratio"
    ]
].copy()

corridor_features.columns = [
    "source_center",
    "destination_center",
    "corridor_trip_count",
    "corridor_delay_ratio"
]

model_df = model_df.merge(
    corridor_features,
    on=["source_center","destination_center"],
    how="left"
)

# Datetime
model_df["trip_creation_time"] = pd.to_datetime(
    model_df["trip_creation_time"],
    format="mixed"
)

# Time features
model_df["trip_hour"] = model_df["trip_creation_time"].dt.hour
model_df["trip_dayofweek"] = model_df["trip_creation_time"].dt.dayofweek
model_df["trip_month"] = model_df["trip_creation_time"].dt.month

# Route type
model_df["route_type"] = model_df["route_type"].map(
    {"FTL":1,"Carting":0}
)

# Engineered features
model_df["distance_per_hour"] = (
    model_df["osrm_distance"]
    /
    model_df["osrm_time"]
)

model_df["network_risk_score"] = (
    model_df["corridor_delay_ratio"]
    *
    (
        model_df["source_betweenness_centrality"]
        +
        model_df["destination_betweenness_centrality"]
    )
)

print(model_df.shape)

model_df.to_csv(
    r"D:\Projects\Delivery_ETA\outputs\model_dataset.csv",
    index=False
)

print("Saved Successfully")

(26368, 29)
Saved Successfully


In [8]:
features = [
    "osrm_time",
    "osrm_distance",
    "actual_distance_to_destination",
    "route_type",

    "source_degree_centrality",
    "source_betweenness_centrality",
    "source_in_degree",
    "source_out_degree",

    "destination_degree_centrality",
    "destination_betweenness_centrality",
    "destination_in_degree",
    "destination_out_degree",

    "corridor_trip_count",
    "corridor_delay_ratio",

    "trip_hour",
    "trip_dayofweek",
    "trip_month",

    "distance_per_hour",
    "network_risk_score"
]

X = model_df[features]

y = model_df["actual_time"]

print(X.shape)

(26368, 19)


In [9]:
missing = X.isna().sum()

print(
    missing[missing > 0]
)

source_degree_centrality               94
source_betweenness_centrality          94
source_in_degree                       94
source_out_degree                      94
destination_degree_centrality          94
destination_betweenness_centrality     94
destination_in_degree                  94
destination_out_degree                 94
corridor_trip_count                   432
corridor_delay_ratio                  432
network_risk_score                    432
dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X = X.fillna(0)

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )
)

print(X_train.shape)
print(X_test.shape)

(21094, 19)
(5274, 19)


In [12]:
from xgboost import XGBRegressor

graph_model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

graph_model.fit(
    X_train,
    y_train
)

print("Training Complete")

Training Complete


## Model Evaluation

In [13]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred = graph_model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE :", round(mae,2))
print("RMSE:", round(rmse,2))
print("R2  :", round(r2,4))

MAE : 31.49
RMSE: 90.92
R2  : 0.9506


In [14]:
pct_error = (
    np.abs(
        y_test - y_pred
    )
    /
    y_test
)

accuracy_15 = (
    (pct_error <= 0.15)
    .mean()
    * 100
)

print(
    f"15% Accuracy: {accuracy_15:.2f}%"
)

15% Accuracy: 54.46%


In [15]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": graph_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
)

feature_importance.head(15)

,feature,importance
1,osrm_distance,0.523344
0,osrm_time,0.244033
2,actual_distance_to_destination,0.102342
13,corridor_delay_ratio,0.046581
3,route_type,0.013282
18,network_risk_score,0.010181
12,corridor_trip_count,0.006674
10,destination_in_degree,0.006411
8,destination_degree_centrality,0.006141
9,destination_betweenness_centrality,0.005910


In [16]:
feature_importance.to_csv(
    r"D:\Projects\Delivery_ETA\outputs\feature_importance.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully
